In [2]:
from dotenv import load_dotenv
import os 
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'true' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'LANG'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])

43b13g4OZS
gpt-4.1-mini
랭스미스로 추적 중입니다 : lsv2_pt_55


- 1단계 : 프롬프트에 부탁
- 2단계 : 가이드라인 - 이런게 스팸, 일반 문자
- 2단계 + a: 어떻게 써야할건지?
- 3단계 : 데이터를 기반으로 판단 
    - 1 : 대표적인 것 몇 개를 선별해서 붙여넣기
    - 2 : 퓨샷 () 의미 + 키워드 BM 25
    - 3 : RAG 

---------------------------- 1주일

머신러닝 모델 + LLM 

---------------------------- 한 달 



In [3]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version="2024-12-01-preview",
    temperature=0.0,
)

from langchain_openai import AzureOpenAIEmbeddings

emb = AzureOpenAIEmbeddings(
    model=EMB_MODEL_NAME,
    api_key=AZURE_OPENAI_EMB_API_KEY,
    azure_endpoint=EMB_END_POINT,
    api_version="2024-08-01-preview"
)

In [5]:
from langchain_core.documents import Document
import pandas as pd

CSV_PATH = "sms_data.csv"
df = pd.read_csv(CSV_PATH, sep="|")
df

,id,subject,label
0,0,[듀] BMW 자유로 전시장 시승 및 상담 안내 김용욱 팀장 연락처 ***-****...,spam
1,1,[Web발신] 케이씨씨오토모빌 일산전시장 VIP 감사 이벤트 안내 *월 한정 특별 ...,spam
2,2,[Web발신](광고)종근당건강 GS홈쇼핑 오늘 오전 방송! 아이커 VIP 특가+사은...,spam
3,3,[Web발신]롯데안산 헤라 특별 할인 안내 남성용 옴므 세트 및 선크림 세트 구매 ...,spam
4,4,"[Web발신] 굿웨어몰 여름 에디션 특가 안내 한기동님, 인기 반팔티와 슬랙스 최대...",spam
...,...,...,...
158,158,[web발신] 고객만족도 설문조사가 진행 중입니다. 참여하신 고객 중 추첨을 통해 ...,ham
159,159,[Web발신] 삼성전자 서비스센터에서는 봄철 에어컨 점검 프로모션을 진행합니다. 점...,ham
160,160,"고객님, 신제품 출시 기념 체험단 신청이 가능합니다. 신청 고객 중 일부는 추첨을 ...",ham
161,161,[web발신] 제휴 카드사에서 여행 경비 지원 이벤트를 진행 중입니다. 참여 조건은...,ham


In [6]:
docs = [
    Document(
        page_content=str(row["subject"]),
        metadata={"id": int(row["id"]), "label": str(row["label"])}
    )
    for _, row in df.iterrows()
]
docs[:2]

[Document(metadata={'id': 0, 'label': 'spam'}, page_content='[듀] BMW 자유로 전시장 시승 및 상담 안내 김용욱 팀장 연락처 ***-****-**** *월 **일(토) *시~**시 방문 예약 필수, 재고 및 조건 문의 환영 경기도 고양시 일산서구 법곳길***번길 **-** [수신거부: ***-***-****]'),
 Document(metadata={'id': 1, 'label': 'spam'}, page_content='[Web발신] 케이씨씨오토모빌 일산전시장 VIP 감사 이벤트 안내 *월 한정 특별 혜택, 현대백화점 킨텍스점 방문 시 발렛파킹권 및 라운지 이용권 제공, 골프 브랜드 할인쿠폰 증정! EXPERIENCE WEEK 시승행사도 함께 진행, 예약 및 문의: 이상규 010-1234-5678. 무료수신거부 080-000-0000')]

In [7]:
from langchain_community.vectorstores import FAISS

#최초 한번만 실행
vectordb = FAISS.from_documents(
    documents=docs,
    embedding=emb
)
vectordb.save_local("faiss_sms_db")

In [8]:
vectordb = FAISS.load_local("faiss_sms_db", emb, allow_dangerous_deserialization=True)

In [9]:
vectordb.similarity_search_with_score('대리님 상품 마감 기한이 오늘까지 입니다.', k=10)

[(Document(id='ab52918e-e80b-4dd5-ab0c-f3cf23f863a9', metadata={'id': 152, 'label': 'ham'}, page_content='[web발신] 새롭게 출시된 적금 상품의 우대금리 신청이 오늘 자정까지 가능합니다. 신청은 앱에서 바로 진행할 수 있습니다.'),
  np.float32(1.0570519)),
 (Document(id='3a95d7ac-1dde-4a33-a285-744e47f3f2e1', metadata={'id': 140, 'label': 'ham'}, page_content='[web발신] 이번 주에만 진행되는 가전제품 보상판매 안내드립니다. 기존 제품 반납 시 추가 보상 혜택이 주어집니다.'),
  np.float32(1.0608046)),
 (Document(id='00d48235-c678-4b4d-8f32-716c9b7f7506', metadata={'id': 87, 'label': 'ham'}, page_content='고객님, 주문하신 상품이 금일 출고되어 내일 도착 예정입니다. 운송장 번호는 문자 하단에서 확인 가능합니다. 문의는 1588-****로 연락주세요.'),
  np.float32(1.1223524)),
 (Document(id='a5dc6844-42df-45ba-8b34-47cd400e4aab', metadata={'id': 94, 'label': 'ham'}, page_content='오늘 오후 3시 고객사 미팅 건입니다. 발표자료는 최신 데이터 기준으로 수정해주세요. 회의 종료 후 피드백 정리 부탁드립니다.'),
  np.float32(1.1298639)),
 (Document(id='c6732ed8-8082-4388-998c-827bbf2c9d38', metadata={'id': 112, 'label': 'ham'}, page_content='고객님, 카드 포인트 유효기간이 이번 달 말까지입니다. 미사용 포인트는 자동 소멸됩니다.'),
  np.float32(

In [13]:
SYSTEM = """너는 스팸 필터 심사관이다.
- 입력된 SMS 제목과, 벡터 유사도로 검색된 '유사 사례들'을 근거로 최종 판단한다.
- 반드시 JSON으로 답하라: {"subject": "입력된 SMS 제목", "label": "spam|ham", "confidence": 0~1, "reason": "<한 줄 근거>"}
- 'label'은 spam 또는 ham 중 하나.
- 'confidence'는 spam이나 ham이라고 판단한 것에 대한 확실도를 0과 1 사이 부동소수(예: 0.78).
- 'rationale'은 핵심 이유 한 줄(예: "광고성 키워드와 단축 URL 포함").
- 유사 사례의 라벨을 무비판적으로 따라하지 말고, 공통 패턴을 근거로 판단하라.
"""

USER_TEMPLATE = """[INPUT_SUBJECT]
{subject}

[RETRIEVED_CASES]
{cases}

[지침]
1) 광고/대출/당첨/의심 링크/수신거부/과도한 혜택/단축 URL/계좌정지 유도 등은 'spam'일 가능성이 높음
2) 가족/지인 대화/회의 일정/은행 정상 알림/업무 커뮤니케이션은 'ham' 가능성이 높음
3) 빈약한 근거면 confidence를 낮게.
4) 최종 출력은 JSON만.
"""

from pydantic import BaseModel, Field
class SMSSpamCase(BaseModel):
    subject: str = Field(description="SMS 제목")
    class : [kkinder | ele]
    label: str = Field(description="SMS 라벨(spam 또는 ham)")
    confidence: float = Field(description="확실도(0~1)")
    reason: str = Field(description="스펨/햄 판단에 대한 한 줄 근거")

from typing import List, Tuple

def retrieve_cases(query: str) -> List[SMSSpamCase]:
    results: List[Tuple[Document, float]] = vectordb.similarity_search_with_score(query, k=5)
    max_d = max([s for _, s in results] + [1e-6])
    for doc, dist in results:
        sim = 1.0 - (dist / max_d)  # 0~1 근사
        doc.metadata["sim"] = sim
    return [d for d, _ in results]

def pretty_cases(cases: List[SMSSpamCase]) -> str:
    lines = []
    for i, c in enumerate(cases, 1):
        lines.append(f"{i}. [label={c.metadata.get('label')} sim≈{c.metadata.get('sim'):.2f}] {c.page_content}")
    return "\n".join(lines)

retrive_results = retrieve_cases('대리님 좋은 상품이 있어 연락드립니다.')
print(retrive_results)
print('--------------------------------')
print(pretty_cases(retrive_results))

[Document(id='00d48235-c678-4b4d-8f32-716c9b7f7506', metadata={'id': 87, 'label': 'ham', 'sim': np.float32(0.05947727)}, page_content='고객님, 주문하신 상품이 금일 출고되어 내일 도착 예정입니다. 운송장 번호는 문자 하단에서 확인 가능합니다. 문의는 1588-****로 연락주세요.'), Document(id='8be1fc1d-974e-4a5d-a9cb-b0c9c49e969d', metadata={'id': 158, 'label': 'ham', 'sim': np.float32(0.027748346)}, page_content='[web발신] 고객만족도 설문조사가 진행 중입니다. 참여하신 고객 중 추첨을 통해 상품권을 드립니다.'), Document(id='2b99b4e7-4b68-4a98-b275-155a28f74bd5', metadata={'id': 13, 'label': 'spam', 'sim': np.float32(0.0024820566)}, page_content='[이마트/드림마트] 초특가 할인 행사 안내! ＊*월**일~*월**일까지 다양한 신선식품 및 생활용품 특가 판매 중! 참외, 골드키위, 한돈앞다리살, 대하새우 등 인기 상품 다수 포함. 주문 및 문의: 이마트 ***-***-****, 드림마트 ***-****-****. 자세한 행사 정보는 http://eventmart.kr/abc123 참고하세요. 수신거부: ***-***-****'), Document(id='6fa6aca1-679d-462b-abb5-d4c5b69cedd4', metadata={'id': 58, 'label': 'spam', 'sim': np.float32(0.0010278821)}, page_content='[Web발신][LG전자 이도본점] 강원숙 고객님, 가정의 달 특별 프로모션 안내드립니다. TV 진열상품 최대 **% 할인, 올레드 및 울트라TV 한정 판매! 구독가전

In [14]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(USER_TEMPLATE)

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": prompt}
]
structured_llm = llm.with_structured_output(SMSSpamCase)

from langchain_core.runnables import RunnablePassthrough, RunnableLambda
run_retrieve = RunnableLambda(retrieve_cases)
run_pretty = RunnableLambda(pretty_cases)

clf_chain = {"subject": RunnablePassthrough(), "cases": run_retrieve | run_pretty } | prompt | structured_llm

In [15]:
import json

if __name__ == "__main__":
    samples = [
        "[Web발신] 하나은행 고객님, 자동이체 만료 예정 안내드립니다. 재등록 시 금리 우대 혜택이 제공됩니다. 자세한 내용은 홈페이지를 참고해주세요.",
        "[Web발신][갤럭시샵 평촌점] 오픈기념 단 5일! S24 시리즈 최대 40만원 혜택, 사은품 증정 및 기기변경 추가 지원. 상담문의 010-3456-7890. 수신거부 080-111-1111.",
        "오늘 오후 2시 사내 회의실 C에서 팀별 진행사항 점검 있습니다. 발표 순서는 공유폴더에서 확인 바랍니다. 참석 필수입니다.",
        "[Web발신][롯데시네마] 주말 예매 고객 대상 팝콘세트 무료 증정 이벤트 진행 중! 영화 예매 시 자동 응모됩니다. 자세한 내용은 앱 공지 참고.",
        "[Web발신][럭키몰] 봄맞이 재고정리 세일! 아이폰·에어팟 최대 70% 할인, 전고객 무료배송. 단 3일 한정, 선착순 소진 시 종료. 바로가기 ▶ bit.ly/springdeal 수신거부 080-000-5555.",
        "대출하신 ‘미드나잇 라이브러리’ 반납 기한이 내일입니다. 연체 시 대출이 제한되오니 유의 바랍니다.",
        "[Web발신][휴대폰샵 신도림점] 번호이동 고객 특별 혜택! 갤럭시, 아이폰, 워치 사전예약 시 사은품 증정. 현금 지원 이벤트 중. 방문상담 환영.",
        "[서울교통공사] 금일 오후 1시~3시 사이 2호선 일부 구간 열차 지연 예정입니다. 승객 여러분의 양해 바랍니다.",
        "[Web발신][현대백화점 무역센터점] 주말 단 3일, VIP 전용 브랜드데이 최대 30% 할인! H포인트 적립까지.",
        "[web발신] 병원 예약 일정이 내일 오전 10시로 변경되었습니다. 변경이 어려우시면 02-234-****로 연락주세요.",
        "[Web발신][골드캐피탈] 급전필요시 당일대출 가능! 신용조회 無, 소득증빙 無, 즉시 송금 진행. 승인율 98%, 상담문의 ☎010-1234-5678. 신청: http://bit.ly/goldloan 수신거부 080-111-2222."
    ]
    for s in samples:
        out = clf_chain.invoke(s)
        jout = json.dumps(out.model_dump(), ensure_ascii=False, indent=2)
        print(jout)
        print('--------------------------------')
        

{
  "subject": "[Web발신] 하나은행 고객님, 자동이체 만료 예정 안내드립니다. 재등록 시 금리 우대 혜택이 제공됩니다. 자세한 내용은 홈페이지를 참고해주세요.",
  "label": "ham",
  "confidence": 0.75,
  "rationale": "메시지는 하나은행에서 자동이체 만료 예정 안내와 재등록 시 금리 우대 혜택을 알리는 정상적인 은행 알림으로 보임. 광고나 의심 링크, 과도한 혜택이 포함되어 있지 않고, 은행 정상 알림에 해당하므로 ham으로 판단."
}
--------------------------------
{
  "subject": "[Web발신][갤럭시샵 평촌점] 오픈기념 단 5일! S24 시리즈 최대 40만원 혜택, 사은품 증정 및 기기변경 추가 지원. 상담문의 010-3456-7890. 수신거부 080-111-1111.",
  "label": "spam",
  "confidence": 0.95,
  "rationale": "문자에 광고성 내용(오픈기념, 최대 40만원 혜택, 사은품 증정, 기기변경 추가 지원)과 상담문의 번호, 수신거부 번호가 포함되어 있어 광고성 스팸 메시지일 가능성이 매우 높음."
}
--------------------------------
{
  "subject": "오늘 오후 2시 사내 회의실 C에서 팀별 진행사항 점검 있습니다. 발표 순서는 공유폴더에서 확인 바랍니다. 참석 필수입니다.",
  "label": "ham",
  "confidence": 0.95,
  "rationale": "내용이 사내 회의 일정 안내로, 업무 커뮤니케이션에 해당하며 광고나 의심 링크가 없으므로 spam 가능성 낮음."
}
--------------------------------
{
  "subject": "[Web발신][롯데시네마] 주말 예매 고객 대상 팝콘세트 무료 증정 이벤트 진행 중! 영화 예매 시 자동 응모됩니다. 자세한 내용은 앱 공지 참고.",
  "label": "ham",
  "co

In [16]:
# 아래 문장들에 대해서 스펨 여부를 판단해 보세요
samples = [
        "[Web발신] 하나은행 고객님, 자동이체 만료 예정 안내드립니다. 재등록 시 금리 우대 혜택이 제공됩니다. 자세한 내용은 홈페이지를 참고해주세요.",
        "[Web발신][갤럭시샵 평촌점] 오픈기념 단 5일! S24 시리즈 최대 40만원 혜택, 사은품 증정 및 기기변경 추가 지원. 상담문의 010-3456-7890. 수신거부 080-111-1111.",
        "오늘 오후 2시 사내 회의실 C에서 팀별 진행사항 점검 있습니다. 발표 순서는 공유폴더에서 확인 바랍니다. 참석 필수입니다.",
        "[Web발신][롯데시네마] 주말 예매 고객 대상 팝콘세트 무료 증정 이벤트 진행 중! 영화 예매 시 자동 응모됩니다. 자세한 내용은 앱 공지 참고.",
        "[Web발신][럭키몰] 봄맞이 재고정리 세일! 아이폰·에어팟 최대 70% 할인, 전고객 무료배송. 단 3일 한정, 선착순 소진 시 종료. 바로가기 ▶ bit.ly/springdeal 수신거부 080-000-5555.",
        "대출하신 ‘미드나잇 라이브러리’ 반납 기한이 내일입니다. 연체 시 대출이 제한되오니 유의 바랍니다.",
        "[Web발신][휴대폰샵 신도림점] 번호이동 고객 특별 혜택! 갤럭시, 아이폰, 워치 사전예약 시 사은품 증정. 현금 지원 이벤트 중. 방문상담 환영.",
        "[서울교통공사] 금일 오후 1시~3시 사이 2호선 일부 구간 열차 지연 예정입니다. 승객 여러분의 양해 바랍니다.",
        "[Web발신][현대백화점 무역센터점] 주말 단 3일, VIP 전용 브랜드데이 최대 30% 할인! H포인트 적립까지.",
        "[web발신] 병원 예약 일정이 내일 오전 10시로 변경되었습니다. 변경이 어려우시면 02-234-****로 연락주세요.",
        "[Web발신][골드캐피탈] 급전필요시 당일대출 가능! 신용조회 無, 소득증빙 無, 즉시 송금 진행. 승인율 98%, 상담문의 ☎010-1234-5678. 신청: http://bit.ly/goldloan 수신거부 080-111-2222."
    ]

In [ ]:
text = input("아무거나 텍스트입력해봐: ")
result = clf_chain.invoke(text)
import json

print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))
